# Capstone: ask each model about one merger

Run the cells from top to bottom. Each section does one job.

Set `DEAL_ID` in section 2. The notebook downloads and queries that deal only. It does not loop over the register.

Leave `DRY_RUN = True` until you want to spend API credits. The last cell is the only one that calls a model.

Keys come from `secrets.env`. Deal ids and filing links come from `M&A Datasets.csv`.


## 1. Setup

Point the notebook at the Capstone folder and import the libraries the later cells use.

In [1]:
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path
import hashlib
import json
import os
import re
import urllib.error
import urllib.request

ROOT = Path.cwd()
if not (ROOT / "secrets.env").exists():
    ROOT = Path("/Users/apizzini/Desktop/Capstone")

SECRETS_FILE = ROOT / "secrets.env"
DEALS_FILE = ROOT / "M&A Datasets.csv"
DATA_DIR = ROOT / "dho_data"

print("Folder:", ROOT)
print("Secrets file:", SECRETS_FILE.name, "(found)" if SECRETS_FILE.exists() else "(missing)")
print("Deal file:", DEALS_FILE.name, "(found)" if DEALS_FILE.exists() else "(missing)")


Folder: /Users/apizzini/Desktop/Capstone
Secrets file: secrets.env (found)
Deal file: M&A Datasets.csv (found)


## 2. Choose one deal

This is the control. Set `DEAL_ID` to one id from `M&A Datasets.csv`. Later cells look up that row's 8-K and DEFM14A links and send only that deal to the models.

`DRY_RUN = True` lists the filings and does not call a model. Set it to `False` when you want the profile and the forecast.

`WHICH_MODELS = None` uses every provider that has a key in `secrets.env`. To use two of them, set it to `["anthropic", "gemini"]`.


In [2]:
# One deal id from the CSV. Examples: ITGR-ARMSTRONG, VAL-RIG, NATH-SFD.
DEAL_ID = "HZO-SAFEHARBOR"

# True = look up the filings only. False = call the models and spend credits.
DRY_RUN = False

# None = every model with a key. Or a list such as ["openai", "anthropic"].
WHICH_MODELS = None

print("This run will use deal id:", DEAL_ID)
print("Dry run:", DRY_RUN)


This run will use deal id: HZO-SAFEHARBOR
Dry run: False


## 3. The two prompts

The first prompt asks a model to write a company profile and a situational analysis from the Form 8-K and Form DEFM14A, and from the Umbrex Top 1000 Global Private Equity Firms directory.

The second prompt is the study question. It sees that profile, not the raw filing.

In [3]:
# Study question. Asked after the model has written its own profile.
FORECAST_PROMPT = (
    "Based on the situation and company profiles given, determine if this "
    "merger and acquisition will be a success or a failure. Give a percentage "
    "of probability for either failed or successful. If the prediction of "
    "success is higher, what is the ROI, WACC and ROA 5 years from acquisition? "
    "If the prediction of failure is higher, give cause, analysis, and "
    "predictive financial outcomes."
)

# Instructions for the profile. The filing text and the PE directory are attached later.
PE_FIRMS_URL = "https://umbrex.com/resources/top-1000-global-private-equity-firms/"

PROFILE_INSTRUCTIONS = """You are reading SEC filings for one proposed merger. The filing text below is the Form 8-K and Form DEFM14A for this deal. Alongside those filings, use the Umbrex Top 1000 Global Private Equity Firms directory at https://umbrex.com/resources/top-1000-global-private-equity-firms/. The directory is attached after the filings. Use it to say whether a named acquirer, parent, or financial sponsor appears on that list, and cite the page when you rely on it. A listing is not a financial figure.

Write two sections with those headings:
1. Company profile
Cover the acquirer and the target: legal names, business, scale, and the capital-structure figures that appear in the filings.
2. Situational analysis
Cover deal structure, per-share consideration, premium, synergies, conditions to closing, termination fees, outside date, required approvals, and the process in the background of the merger.

Cite the form type for every material number from the filings. If a number is not in the filing text, write "missing from the filings provided." Do not invent figures.
"""


def _fetch_pe_page():
    """Download the Umbrex directory page. The site blocks a plain Python client."""
    import subprocess

    request = urllib.request.Request(
        PE_FIRMS_URL,
        headers={"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"},
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return response.read().decode("utf-8", "replace")
    except (urllib.error.URLError, TimeoutError):
        completed = subprocess.run(
            ["curl", "-fsSL", "-A", "Mozilla/5.0", PE_FIRMS_URL],
            check=True,
            capture_output=True,
        )
        return completed.stdout.decode("utf-8", "replace")


def pe_directory_text():
    """Firm names from the Umbrex Top 1000 page. Cached so each run sees the same list."""
    cache = DATA_DIR / "umbrex_top_1000_pe_firms.txt"
    if cache.is_file():
        names = cache.read_text().strip()
    else:
        html = _fetch_pe_page()
        match = re.search(r"FIRMS=`(.*?)`", html, re.S)
        if not match:
            raise RuntimeError(f"Could not read the firm list from {PE_FIRMS_URL}")
        names = "\n".join(
            line.split("\t", 1)[0].strip()
            for line in match.group(1).splitlines()
            if line.strip()
        )
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_text(names + "\n")
    return (
        "Umbrex Top 1000 Global Private Equity Firms\n"
        f"Source: {PE_FIRMS_URL}\n"
        "One firm name per line:\n"
        + names
    )


def profile_prompt_for(packet):
    """Filings plus the Umbrex directory, under the profile instructions."""
    return (
        PROFILE_INSTRUCTIONS
        + "\n\nFiling text:\n\n"
        + packet
        + "\n\nReference directory:\n\n"
        + pe_directory_text()
    )


## 4. Load model keys

Reads `secrets.env`. A blank key skips that model. The key itself is never printed.

`configured_models` returns only the providers that have a key.

In [4]:
def load_secrets(path):
    """Copy KEY=value lines from secrets.env into the environment."""
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Copy secrets.env.example and add your keys.")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value


def configured_models():
    """List the models we can call. Skip any provider whose key is blank."""
    catalog = [
        ("openai", "OPENAI_API_KEY", "OPENAI_MODEL", "gpt-4.1", "openai", "https://api.openai.com/v1/chat/completions"),
        ("anthropic", "ANTHROPIC_API_KEY", "ANTHROPIC_MODEL", "claude-sonnet-5-5", "anthropic", "https://api.anthropic.com/v1/messages"),
        ("xai", "XAI_API_KEY", "XAI_MODEL", "grok-3", "openai", "https://api.x.ai/v1/chat/completions"),
        ("gemini", "GEMINI_API_KEY", "GEMINI_MODEL", "gemini-3.1-pro-preview", "gemini", "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"),
    ]
    ready = []
    for name, key_env, model_env, default_model, kind, url in catalog:
        key = os.environ.get(key_env, "").strip()
        if not key:
            print(f"  {name}: skipped, no {key_env}")
            continue
        model = os.environ.get(model_env, default_model).strip() or default_model
        ready.append({"name": name, "key": key, "model": model, "kind": kind, "url": url})
        print(f"  {name}: {model}")
    return ready


load_secrets(SECRETS_FILE)
MODELS = configured_models()


  openai: gpt-4.1
  anthropic: claude-sonnet-5-5
  xai: grok-3
  gemini: gemini-3.1-pro-preview


## 5. Load the deal register

`load_deals` reads `M&A Datasets.csv`. One row is one merger. The 8-K and DEFM14A columns hold the SEC links used to download the filings. A cell may contain more than one link.

In [5]:
def split_urls(cell):
    """Pull every https link out of a spreadsheet cell."""
    return re.findall(r"https://\S+", cell or "")


def form_from_url(url):
    """Guess the form type from the file name in the EDGAR link."""
    name = url.lower()
    if "defm14a" in name:
        return "DEFM14A"
    if "8k" in name or "8-k" in name:
        return "8-K"
    return "filing"


def load_deals(path):
    """Return one dict per deal: id, name, and the filing links."""
    import csv
    deals = []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            filings = []
            for column in ("8-K", "DEFM14A"):
                for url in split_urls(row.get(column, "")):
                    filings.append({"column": column, "form": form_from_url(url), "url": url})
            deals.append({
                "deal_id": row["Deal id"].strip(),
                "name": row["Deal"].strip(),
                "filings": filings,
            })
    return deals


def show_deals(deals):
    """Print the register and flag a link that sits in the wrong column."""
    for deal in deals:
        print(f"{deal['deal_id']:18} {deal['name']}")
        for filing in deal["filings"]:
            note = ""
            if filing["column"] == "8-K" and filing["form"] == "DEFM14A":
                note = "  <-- this 8-K cell points at a proxy"
            if filing["column"] == "DEFM14A" and filing["form"] == "8-K":
                note = "  <-- this proxy cell points at an 8-K"
            print(f"  {filing['column']:8} {filing['url']}{note}")


DEALS = load_deals(DEALS_FILE)
show_deals(DEALS)


def deal_by_id(deal_id):
    """Find one deal in the register."""
    for deal in DEALS:
        if deal["deal_id"] == deal_id:
            return deal
    known = ", ".join(deal["deal_id"] for deal in DEALS)
    raise KeyError(f"No deal id {deal_id}. Choose one of: {known}")


ITGR-ARMSTRONG     Integer Holdings (ITGR) / KKR affiliate (Armstrong)
  8-K      https://www.sec.gov/Archives/edgar/data/1114483/000095010326011881/dp251220_8k.htm
  DEFM14A  https://www.sec.gov/Archives/edgar/data/1114483/000114036126036420/ny20080277x2_defm14a.htm
WEAV-FP            Weave Communications (WEAV) / Francisco Partners
  8-K      https://www.sec.gov/Archives/edgar/data/1609151/000160915126000100/weav-20260818.htm
  DEFM14A  https://www.sec.gov/Archives/edgar/data/1609151/000114036126037628/ny20081403x2_defm14a.htm
NATH-SFD           Nathan’s Famous (NATH) / Smithfield Foods
  8-K      https://www.sec.gov/Archives/edgar/data/1609151/000114036126037628/ny20081403x2_defm14a.htm  <-- this 8-K cell points at a proxy
  DEFM14A  https://www.sec.gov/Archives/edgar/data/69733/000110465926110393/tm267612-2_defm14a.htm
VREX-TDY           Varex Imaging (VREX) / Teledyne
  8-K      https://www.sec.gov/Archives/edgar/data/1681622/000110465926093349/tm2622593d1_8k.htm
  DEFM14A  https:

## 6. Turn a filing into text

`html_to_text` drops scripts and tags. `clip_text` keeps the filing under a character cap so a long proxy does not blow the context window. The start of the document is kept, then windows around the sections the study cares about.

In [6]:
class _TextExtractor(HTMLParser):
    """Collect visible text and skip script and style blocks."""
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self._skip = 0

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style", "noscript"}:
            self._skip += 1

    def handle_endtag(self, tag):
        if tag in {"script", "style", "noscript"} and self._skip:
            self._skip -= 1
        if tag in {"p", "div", "br", "tr", "li", "h1", "h2", "h3", "h4"}:
            self.parts.append("\n")

    def handle_data(self, data):
        if not self._skip:
            self.parts.append(data)


def html_to_text(html):
    """Convert filing HTML into plain text."""
    parser = _TextExtractor()
    parser.feed(html)
    lines = [" ".join(line.split()) for line in "".join(parser.parts).splitlines()]
    return "\n".join(line for line in lines if line)


SECTION_LABELS = [
    "summary",
    "background of the merger",
    "reasons",
    "opinion of",
    "financing",
    "regulatory",
    "merger agreement",
]


def clip_text(text, max_chars):
    """Shorten a long filing. Returns the excerpt and whether it was cut."""
    if len(text) <= max_chars:
        return text, False
    head = text[: max_chars // 5]
    windows = [head]
    used = len(head)
    lowered = text.lower()
    span = max(4000, (max_chars - used) // len(SECTION_LABELS))
    for label in SECTION_LABELS:
        at = lowered.find(label)
        if at < 0:
            continue
        piece = text[max(0, at - 500) : at + span]
        windows.append(f"\n\n[excerpt around '{label}']\n{piece}")
        used += len(piece)
        if used >= max_chars:
            break
    return "".join(windows)[:max_chars], True


## 7. Download the 8-K and the DEFM14A

`download_filing` saves the raw HTML under `dho_data/raw` and names the file with a short hash. `prepare_packet` downloads every link for one deal and builds the text that will go to the models.

In [7]:
def download_filing(url, dest):
    """Download one EDGAR page. Requires SEC_USER_AGENT in secrets.env."""
    agent = os.environ.get("SEC_USER_AGENT", "").strip()
    if not agent:
        raise RuntimeError("Set SEC_USER_AGENT in secrets.env before downloading filings.")
    request = urllib.request.Request(url, headers={"User-Agent": agent, "Accept-Encoding": "gzip, deflate"})
    with urllib.request.urlopen(request, timeout=60) as response:
        raw = response.read()
        if response.headers.get("Content-Encoding") == "gzip":
            import gzip
            raw = gzip.decompress(raw)
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(raw)
    return raw


def prepare_packet(deal, max_chars):
    """Download the deal's filings and return the text packet plus a short log."""
    blocks = []
    log = []
    for filing in deal["filings"]:
        digest = hashlib.sha256(filing["url"].encode()).hexdigest()[:12]
        path = DATA_DIR / "raw" / f"{deal['deal_id']}_{filing['form']}.{digest}.html"
        if path.exists():
            raw = path.read_bytes()
        else:
            raw = download_filing(filing["url"], path)
        text = html_to_text(raw.decode("utf-8", errors="replace"))
        clipped, truncated = clip_text(text, max_chars)
        blocks.append(f"===== {filing['form']} {path.name} =====\n{clipped}")
        log.append({
            "form": filing["form"],
            "url": filing["url"],
            "chars": len(text),
            "sent": len(clipped),
            "truncated": truncated,
        })
        flag = " truncated" if truncated else ""
        print(f"  {filing['form']}: {len(text):,} chars extracted, {len(clipped):,} sent{flag}")
    return "\n\n".join(blocks), log


## 8. Call a model and price the call

`ask_model` sends one prompt and returns the text plus the token counts from the API.

`price_call` turns those counts into dollars. Rates are list prices checked on 2026-10-19. A model that is not in the table still runs; its dollar amount is left blank.

In [8]:
# USD per 1,000,000 tokens. long_* replaces the base rate once the prompt exceeds long_after tokens.
PRICE_PER_MILLION = {
    "gpt-4.1": {"input": 2.00, "output": 8.00, "cached_input": 0.50},
    "claude-sonnet-5-5": {"input": 2.00, "output": 10.00, "cached_input": 0.10, "cache_write": 2.50},
    "claude-sonnet-4-5": {"input": 3.00, "output": 15.00, "cached_input": 0.30, "cache_write": 3.75},
    "grok-3": {"input": 3.00, "output": 15.00},
    "grok-4.7": {"input": 2.00, "output": 6.00, "cached_input": 0.50, "long_after": 200_000, "long_input": 4.00, "long_output": 12.00},
    "gemini-3.1-pro-preview": {"input": 2.00, "output": 12.00, "cached_input": 0.20, "long_after": 200_000, "long_input": 4.00, "long_output": 18.00},
}


def lookup_rate(model):
    """Find the price row for a model id, ignoring a trailing date stamp."""
    key = model.lower().strip()
    if key in PRICE_PER_MILLION:
        return PRICE_PER_MILLION[key]
    trimmed = key.rsplit("-", 1)[0] if len(key) > 9 and key[-8:].isdigit() else key
    matches = [name for name in PRICE_PER_MILLION if trimmed == name or trimmed.startswith(name)]
    if not matches:
        return None
    return PRICE_PER_MILLION[max(matches, key=len)]


def price_call(model, usage):
    """Dollar cost of one prompt from the API's token counts."""
    rate = lookup_rate(model)
    incoming = int(usage.get("input_tokens") or 0)
    cached = int(usage.get("cached_input_tokens") or 0)
    written = int(usage.get("cache_write_tokens") or 0)
    outgoing = int(usage.get("output_tokens") or 0)
    if rate is None:
        return {"input_tokens": incoming, "output_tokens": outgoing, "usd": None}
    input_rate, output_rate = rate["input"], rate["output"]
    cached_rate = rate.get("cached_input", input_rate)
    write_rate = rate.get("cache_write", input_rate)
    if rate.get("long_after") and (incoming + cached + written) > rate["long_after"]:
        input_rate = rate.get("long_input", input_rate)
        output_rate = rate.get("long_output", output_rate)
    usd = (incoming * input_rate + cached * cached_rate + written * write_rate + outgoing * output_rate) / 1_000_000
    return {"input_tokens": incoming, "cached_input_tokens": cached, "output_tokens": outgoing, "usd": round(usd, 6)}


def post_json(url, payload, headers, timeout):
    """POST JSON and return the parsed body. The error text never includes the API key."""
    data = json.dumps(payload).encode()
    request = urllib.request.Request(url, data=data, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode())
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(f"HTTP {exc.code}: {body}") from exc


def usage_from(kind, body):
    """Read token counts from an OpenAI, Anthropic, or Gemini response."""
    if kind == "openai":
        usage = body.get("usage") or {}
        prompt = int(usage.get("prompt_tokens") or 0)
        cached = int((usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0)
        return {"input_tokens": max(prompt - cached, 0), "cached_input_tokens": cached, "output_tokens": int(usage.get("completion_tokens") or 0)}
    if kind == "anthropic":
        usage = body.get("usage") or {}
        return {
            "input_tokens": int(usage.get("input_tokens") or 0),
            "cached_input_tokens": int(usage.get("cache_read_input_tokens") or 0),
            "cache_write_tokens": int(usage.get("cache_creation_input_tokens") or 0),
            "output_tokens": int(usage.get("output_tokens") or 0),
        }
    usage = body.get("usageMetadata") or {}
    thoughts = int(usage.get("thoughtsTokenCount") or 0)
    return {
        "input_tokens": int(usage.get("promptTokenCount") or 0),
        "cached_input_tokens": int(usage.get("cachedContentTokenCount") or 0),
        "output_tokens": int(usage.get("candidatesTokenCount") or 0) + thoughts,
    }


def ask_model(provider, prompt, max_tokens, timeout):
    """Send one prompt to one provider. Returns the reply text and token counts."""
    if provider["kind"] == "openai":
        body = post_json(
            provider["url"],
            {
                "model": provider["model"],
                "temperature": 0,
                "max_tokens": max_tokens,
                "messages": [{"role": "user", "content": prompt}],
            },
            {"Authorization": f"Bearer {provider['key']}", "Content-Type": "application/json"},
            timeout,
        )
        content = body["choices"][0]["message"]["content"]
        if isinstance(content, list):
            content = "\n".join(part.get("text", "") for part in content if isinstance(part, dict))
        return content, usage_from("openai", body)
    if provider["kind"] == "anthropic":
        body = post_json(
            provider["url"],
            {"model": provider["model"], "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]},
            {"x-api-key": provider["key"], "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
            timeout,
        )
        text = "\n".join(block.get("text", "") for block in body.get("content", []))
        return text, usage_from("anthropic", body)
    url = provider["url"].format(model=provider["model"])
    body = post_json(
        url,
        {"contents": [{"role": "user", "parts": [{"text": prompt}]}], "generationConfig": {"temperature": 0, "maxOutputTokens": max_tokens}},
        {"x-goog-api-key": provider["key"], "Content-Type": "application/json"},
        timeout,
    )
    parts = body["candidates"][0]["content"]["parts"]
    text = "\n".join(part.get("text", "") for part in parts)
    return text, usage_from("gemini", body)


## 9. Profile, then forecast

`run_provider` makes two calls. The first writes the company profile from the 8-K, the DEFM14A, and the Umbrex private-equity directory. The second answers the success-or-failure question using only that profile.

`run_deal` does this for every selected model and writes `report.md`.

In [9]:
def money(usd):
    """Format a dollar amount. Blank when the model has no stored price."""
    return "n/a" if usd is None else f"${usd:,.4f}"


def run_provider(provider, packet, max_tokens, timeout):
    """Ask one model for a profile, then for the success or failure forecast."""
    result = {"provider": provider["name"], "model": provider["model"], "calls": []}
    profile_prompt = profile_prompt_for(packet)
    try:
        profile, usage = ask_model(provider, profile_prompt, max_tokens, timeout)
    except Exception as exc:
        result["error"] = str(exc)
        return result
    result["profile"] = profile
    result["calls"].append({"prompt": "profile", **price_call(provider["model"], usage)})

    forecast_prompt = "Company profile and situational analysis:\n\n" + profile + "\n\n" + FORECAST_PROMPT
    try:
        forecast, usage = ask_model(provider, forecast_prompt, max_tokens, timeout)
    except Exception as exc:
        result["error"] = str(exc)
        return result
    result["forecast"] = forecast
    result["calls"].append({"prompt": "forecast", **price_call(provider["model"], usage)})
    return result


def write_report(folder, deal, results):
    """Save a readable report and one JSON file per model."""
    folder.mkdir(parents=True, exist_ok=True)
    lines = [f"# {deal['name']}", "", f"Deal id: {deal['deal_id']}", "", "## Cost", ""]
    total = 0.0
    priced = False
    for result in results:
        (folder / f"{result['provider']}.json").write_text(json.dumps(result, indent=2))
        for call in result.get("calls") or []:
            if call["usd"] is not None:
                total += call["usd"]
                priced = True
            lines.append(
                f"- {result['provider']} {result['model']} {call['prompt']}: "
                f"in {call['input_tokens']:,} out {call['output_tokens']:,} {money(call['usd'])}"
            )
    if priced:
        lines.append(f"- All models: {money(total)}")
    lines += ["", "## Forecasts", ""]
    for result in results:
        lines.append(f"### {result['provider']} ({result['model']})")
        lines.append("")
        if result.get("profile"):
            lines += ["#### Company profile and situational analysis", "", result["profile"], ""]
        if result.get("forecast"):
            lines += ["#### Success or failure", "", result["forecast"], ""]
        if result.get("error"):
            lines += [f"Error: {result['error']}", ""]
    path = folder / "report.md"
    path.write_text("\n".join(lines))
    return path


def run_deal(deal_id, model_names=None, max_chars=80000, max_tokens=16000, timeout=600, dry_run=False):
    """Download one deal and ask each selected model for a profile and a forecast."""
    deal = deal_by_id(deal_id)
    selected = MODELS
    if model_names:
        wanted = {name.strip() for name in model_names}
        selected = [item for item in MODELS if item["name"] in wanted]
    print(deal["name"], f"({deal['deal_id']})")
    print("Models:", ", ".join(f"{item['name']} {item['model']}" for item in selected) or "none")
    if dry_run:
        print("Dry run. Filings that would be sent:")
        for filing in deal["filings"]:
            print(f"  {filing['form']} {filing['url']}")
        return None
    packet, _log = prepare_packet(deal, max_chars)
    results = []
    for provider in selected:
        print(f"Calling {provider['name']}...")
        result = run_provider(provider, packet, max_tokens, timeout)
        if result.get("error"):
            print(f"  failed: {result['error']}")
        for call in result.get("calls") or []:
            print(f"  {call['prompt']}: in {call['input_tokens']:,} out {call['output_tokens']:,} {money(call['usd'])}")
        results.append(result)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    report = write_report(DATA_DIR / "llm_runs" / deal_id / stamp, deal, results)
    print("Wrote", report)
    return report


## 10. Run this deal

Uses the `DEAL_ID` you set in section 2. Nothing else in the register is downloaded or sent to a model.

With `DRY_RUN = True` this cell only prints that deal's filing links. Set `DRY_RUN = False` in section 2, then run this cell, to call the models.


In [10]:
print("Running one deal:", DEAL_ID)
run_deal(DEAL_ID, model_names=WHICH_MODELS, dry_run=DRY_RUN)

Running one deal: HZO-SAFEHARBOR
MarineMax (HZO) / Safe Harbor (Blackstone Infrastructure) (HZO-SAFEHARBOR)
Models: openai gpt-4.1, anthropic claude-sonnet-5-5, xai grok-3, gemini gemini-3.1-pro-preview
  8-K: 34,995 chars extracted, 34,995 sent
  DEFM14A: 914,126 chars extracted, 80,000 sent truncated
Calling openai...
  profile: in 29,417 out 1,351 $0.0696
  forecast: in 1,441 out 1,462 $0.0146
Calling anthropic...
  profile: in 50,233 out 4,096 $0.1414
  forecast: in 2,751 out 3,281 $0.0383
Calling xai...
  profile: in 28,691 out 802 $0.0985
  forecast: in 948 out 161 $0.0056
Calling gemini...
  profile: in 29,873 out 3,437 $0.1010
  forecast: in 1,102 out 2,857 $0.0365
Wrote /Users/apizzini/Desktop/Capstone/dho_data/llm_runs/HZO-SAFEHARBOR/20261106T152538Z/report.md


PosixPath('/Users/apizzini/Desktop/Capstone/dho_data/llm_runs/HZO-SAFEHARBOR/20261106T152538Z/report.md')